[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# An Event Store &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, which builds the `store` and `summary` tables and the
trigger and empties them. Run it first. Each task builds on the rows the ones before it wrote, so
these are worth running in order.


In [1]:
import asyncio
import getpass
import json
import logging
import os
import subprocess
import sys
import threading
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
import psycopg_pool

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

logging.getLogger("psycopg.pool").setLevel(logging.CRITICAL)        # its retries are not the lesson


def build_store():
    """The two tables and the trigger this notebook is about. Idempotent, and it empties them."""
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        conn.execute("""CREATE TABLE IF NOT EXISTS store (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS summary (
                            kind text PRIMARY KEY,
                            n bigint NOT NULL,
                            through bigint NOT NULL)""")            # the watermark, stored
        conn.execute("""CREATE OR REPLACE FUNCTION store_landed() RETURNS trigger AS $$
                        BEGIN
                            PERFORM pg_notify('store', NEW.kind);
                            RETURN NEW;
                        END;
                        $$ LANGUAGE plpgsql""")
        conn.execute("""CREATE OR REPLACE TRIGGER store_announced AFTER INSERT ON store
                        FOR EACH ROW EXECUTE FUNCTION store_landed()""")
        conn.execute("TRUNCATE store, summary")
        return "store, summary and the trigger are ready"


def events(count, kind="click", start=0):
    """Rows to load, as tuples, which is what COPY wants."""
    return [(kind, json.dumps({"n": number, "size": 1 + number % 7}))
            for number in range(start, start + count)]


def counted(sql="SELECT count(*) FROM store", parameters=None):
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        return conn.execute(sql, parameters).fetchone()[0]


print("server:", start_server())
print("events table:", build(), "rows")
print(build_store())


server: already running
events table: 5000 rows
store, summary and the trigger are ready


**1.** A thousand rows, in one statement.


In [2]:
def load(rows):
    with psycopg.connect("dbname=guide") as conn:
        with conn.cursor().copy("COPY store (kind, payload) FROM STDIN") as copy:
            for row in rows:
                copy.write_row(row)
        conn.commit()


load(events(1000))
print("in the table:", counted())


in the table: 1000


One statement, one transaction, and one commit. Everything else in this notebook assumes the rows
arrive this way, which is why the notification behavior further down matters.


**2.** Every kind, in one pipeline.


In [3]:
load(events(400, kind="view"))

with psycopg.connect("dbname=guide") as conn:
    high = conn.execute("SELECT coalesce(max(id), 0) FROM store").fetchone()[0]
    totals = conn.execute("SELECT kind, count(*) FROM store WHERE id <= %s GROUP BY kind",
                          (high,)).fetchall()
    with conn.pipeline():
        for kind, number in totals:
            conn.execute("""INSERT INTO summary (kind, n, through) VALUES (%s, %s, %s)
                            ON CONFLICT (kind) DO UPDATE
                            SET n = EXCLUDED.n, through = EXCLUDED.through""",
                         (kind, number, high))
    conn.commit()

print("summary through id", high, ":", totals)


summary through id 1400 : [('click', 1000), ('view', 400)]


Reading `max(id)` first is what makes the summary describe a definite prefix of the table. Without
it, each statement would see whatever had been committed by the time it ran.


**3.** Three questions, one wait.


In [4]:
pool = await asyncpg.create_pool(database="guide", min_size=3, max_size=3)

start = time.perf_counter()
rows, kinds, newest = await asyncio.gather(
    pool.fetchval("SELECT count(*) FROM store"),
    pool.fetch("SELECT kind, n FROM summary ORDER BY kind"),
    pool.fetchval("SELECT max(id) FROM store"))

print("rows:", rows, "| newest id:", newest)
print("summary:", [tuple(k) for k in kinds])
print(f"one wait, {time.perf_counter() - start:.1f}s")
await pool.close()


rows: 1400 | newest id: 1400
summary: [('click', 1000), ('view', 400)]
one wait, 0.0s


Three connections out of a pool of three, and the whole thing takes as long as the slowest one
rather than the sum of the three.


**4.** How many notifications five hundred identical rows make.


In [5]:
listener = psycopg.connect("dbname=guide", autocommit=True)
listener.execute("LISTEN store")

before = counted("SELECT coalesce(max(id), 0) FROM store")
threading.Timer(0.3, load, args=(events(500, kind="all_the_same"),)).start()
notifications = list(listener.notifies(timeout=4, stop_after=500))

print("rows written: ", counted("SELECT coalesce(max(id), 0) FROM store") - before)
print("notifications:", len(notifications))
listener.execute("UNLISTEN *")
listener.close()


rows written:  500
notifications: 1


One. Identical notifications raised inside a single transaction are delivered once, so the count
says nothing about how many rows arrived.


**5.** Everything after a watermark.


In [6]:
watermark = counted("SELECT coalesce(max(id), 0) FROM store") - 500

conn = await asyncpg.connect(database="guide")
rows = await conn.fetch("SELECT id, kind FROM store WHERE id > $1 ORDER BY id", watermark)
await conn.close()

print("from id", watermark, "found", len(rows), "rows")
print("the new watermark is", rows[-1]["id"] if rows else watermark)


from id 1400 found 500 rows
the new watermark is 1900


The same query answers both "I was just woken" and "I have just restarted", which is what makes a
missed notification survivable.


**6.** How far behind the summary is.


In [7]:
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    rows = conn.execute("SELECT count(*) FROM store").fetchone()[0]
    through = conn.execute("SELECT coalesce(max(through), 0) FROM summary").fetchone()[0]
    behind = conn.execute("SELECT count(*) FROM store WHERE id > %s", (through,)).fetchone()[0]

print("rows in the store:      ", rows)
print("summarized through id:  ", through)
print("not yet summarized:     ", behind)


rows in the store:       1900
summarized through id:   1400
not yet summarized:      500


That last number is the one to put on a dashboard. It rises when loading outpaces summarizing and
returns to zero when the summary catches up, and it is computed without any notification at all.


---

&#8592; **Back to:** [An Event Store](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/17-an-event-store.ipynb)  &nbsp;&middot;&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
